# Pitaco — fine-tuning a 4B student for Portuguese aspect-based sentiment

Kaggle version of the main experiment from [github.com/PedroMLegnaghi/pitaco](https://github.com/PedroMLegnaghi/pitaco). It rebuilds the data splits from the Olist dataset (attached as input), checks they match the split IDs committed in the repo, fine-tunes `Qwen/Qwen3.5-4B` with 16-bit LoRA on teacher-labeled (silver) reviews, and predicts the hand-labeled gold sets.

**Accelerator: GPU T4** (not P100 — too old for the Triton kernels used here).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone --depth 1 https://github.com/PedroMLegnaghi/pitaco.git
%cd pitaco

In [ ]:
!pip install -q -r requirements/kaggle-gpu.txt
!pip install -q --no-deps --ignore-requires-python -e .
# Optimized kernel for Qwen3.5's linear-attention layers. Without it generation falls back to a
# reference PyTorch implementation (correct but several times slower) — latency numbers depend on it.
!pip install -q causal-conv1d || echo 'causal-conv1d unavailable: reference kernel will be used (slower)'
!python -c "import importlib.util as u; print('causal_conv1d active:', u.find_spec('causal_conv1d') is not None)"

Experiment tracking: add a Kaggle secret named `WANDB_API_KEY` (Add-ons → Secrets) to log to Weights & Biases; without it the run logs offline.

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    os.environ["WANDB_MODE"] = "offline"
    print("W&B secret not found: logging offline")

Rebuild the splits from the raw CSV and verify they are identical to the committed ones (`git diff` must be empty).

In [ ]:
import glob, os, shutil
found = glob.glob('/kaggle/input/**/olist_order_reviews_dataset.csv', recursive=True)
assert found, 'Olist not attached: right panel → Add Input → search "Brazilian E-Commerce Public Dataset by Olist"'
os.makedirs('data/raw', exist_ok=True)
shutil.copy(found[0], 'data/raw/')
print('using', found[0])

In [ ]:
!pitaco data --config configs/data.yaml
!git diff --stat --exit-code data/splits && echo 'splits identical to the repo'

In [ ]:
!pitaco train-student --config configs/student/qwen35_4b_lora.yaml

Predictions on gold dev/test: the base model zero-shot (before fine-tuning) and the fine-tuned adapter. Both measure batch-1 latency and batched throughput on the T4.

In [ ]:
!pitaco predict-student --config configs/student/qwen35_4b_lora.yaml
!pitaco predict-student --config configs/student/qwen35_4b_lora.yaml --adapter outputs/student-qwen35-4b/adapter

In [ ]:
!tar czf /kaggle/working/pitaco_outputs.tgz outputs/student-qwen35-4b/adapter results/predictions results/efficiency
!ls -lh /kaggle/working